In [9]:
# load Libraries
import zipfile
import os
from pathlib import Path
from dotenv import load_dotenv  # type: ignore
from kaggle.api.kaggle_api_extended import KaggleApi  # type: ignore
import re
from collections import Counter

In [5]:
# Explicitly point to the .env file relative to this notebook
env_path = Path(__file__).parent / '.env' if '__file__' in dir() else Path('.env')
load_dotenv(dotenv_path=env_path)

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

In [6]:
# Create the 'raw' directory if it doesn't exist
Path("raw").mkdir(exist_ok=True)

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "yorkyong/text8-zip",
    path="raw",
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/yorkyong/text8-zip


In [7]:
# Read the text8 dataset and tokenize it
with open("raw/text8", "r", encoding="utf-8") as f:
    text = f.read()

# Preprocess the text: convert to lowercase and remove punctuation
text = text.lower()
text = re.sub(r"[^a-z\s]", "", text) 

tokens = text.split()
print(len(tokens))
print(tokens[:20])

17005207
['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english']


In [11]:
# Analyze word frequencies
word_counts = Counter(tokens)

print("Total tokens:", len(tokens))
print("Total unique words:", len(word_counts))
print("Top 10 words:", word_counts.most_common(10))

Total tokens: 17005207
Total unique words: 253854
Top 10 words: [('the', 1061396), ('of', 593677), ('and', 416629), ('one', 411764), ('in', 372201), ('a', 325873), ('to', 316376), ('zero', 264975), ('nine', 250430), ('two', 192644)]


In [ ]:
# Filter out infrequent words
min_count = 5
tokens = [w for w in tokens if word_counts[w] >= min_count]

In [ ]:
# Create word-to-index and index-to-word mappings
word2idx = {w: i for i, w in enumerate(word_counts.keys())}
idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)
print(vocab_size)

253854


In [ ]:
# Convert tokens to indices
indexed_tokens = [word2idx[w] for w in tokens]

In [ ]:
# Create training pairs for the skip-gram model
window_size = 2
pairs = []

for i in range(window_size, len(indexed_tokens) - window_size):
    target = indexed_tokens[i]

    for j in range(-window_size, window_size + 1):
        if j != 0:
            context = indexed_tokens[i + j]
            pairs.append((target, context))

print(len(pairs))

66875360


In [ ]:
Path("processed").mkdir(exist_ok=True)